# 2 · Similarity Measures and Their Blind Spots

*[ensemble-codesim](https://github.com/jorge-martinez-gil/ensemble-codesim) tutorial series.*

A **similarity measure** maps a pair of code snippets to a number in `[0, 1]`. The production repo ships **21** of them; here we use a transparent, dependency-free subset (`tutorials/codesim_edu.py`) so we can read every line.

The measures fall into families:

| Family | Idea | Here |
|--------|------|------|
| **Token-based** | Compare bags / sets / n-grams of tokens | `jaccard_tokens`, `ngram_cosine` |
| **Sequence / edit** | Align the two token streams | `sequence_ratio`, `levenshtein_ratio`, `lcs_ratio` |
| **Structural (AST proxy)** | Compare token *kinds*, not names | `token_kind_cosine` |
| **Rename-robust** | Normalise identifiers first | `normalized_jaccard` |
| **Software metrics** | Compare size / control-flow shape | `metrics_cosine` |

### Learning objectives
1. See how each family scores the four clone types.
2. Identify each measure's **blind spot**.
3. Measure **complementarity** — do measures make the *same* mistakes? (No — and that is the whole point.)

In [ ]:
# --- setup: make the tutorial modules importable from anywhere in the repo ---
import os, sys
def _find_tutorials(start):
    d = os.path.abspath(start)
    for _ in range(8):
        if os.path.exists(os.path.join(d, "codesim_edu.py")):
            return d
        if os.path.exists(os.path.join(d, "tutorials", "codesim_edu.py")):
            return os.path.join(d, "tutorials")
        nd = os.path.dirname(d)
        if nd == d: break
        d = nd
    raise RuntimeError("Run this notebook from inside the ensemble-codesim repo.")
TUT  = _find_tutorials(os.getcwd())
REPO = os.path.dirname(TUT)
DATA = os.path.join(REPO, "outputs")
sys.path.insert(0, TUT); sys.path.insert(0, os.path.join(TUT, "examples"))
print("tutorials:", TUT)

In [ ]:
import numpy as np
from codesim_edu import MEASURES, all_scores
from clone_pairs import load_pairs

pairs = load_pairs()
measure_names = list(MEASURES)
print("Measures:", measure_names)

# Sanity peek: how do the measures disagree on a single Type-2 (renamed) pair?
p2 = next(p for p in pairs if p["id"] == "java_type2_sum_renamed")
print("\nScores for a Type-2 (renamed) clone:")
for k, v in all_scores(p2["code_a"], p2["code_b"]).items():
    print(f"  {k:20s} {v:.3f}")

## The measure × pair matrix

We score **every measure** on **every example pair**. Reading across a row shows how one pair is seen by different lenses; reading down a column shows one measure's behaviour across clone types.

In [ ]:
rows = []
labels = []
types = []
ids = []
for p in pairs:
    s = all_scores(p["code_a"], p["code_b"])
    rows.append([s[m] for m in measure_names])
    labels.append(p["label"])
    types.append(p["clone_type"])
    ids.append(p["id"])
M = np.array(rows)            # shape (n_pairs, n_measures)
labels = np.array(labels)

# pretty print
hdr = "pair".ljust(32) + "ty  lbl  " + " ".join(n[:7].rjust(7) for n in measure_names)
print(hdr); print("-" * len(hdr))
for i, p in enumerate(pairs):
    ty = f"T{types[i]}" if types[i] is not None else "neg"
    cells_ = " ".join(f"{M[i,j]:7.2f}" for j in range(len(measure_names)))
    print(f"{ids[i]:32s}{ty:4s}{labels[i]:<5d}{cells_}")

## Visualising the matrix (optional)

If `matplotlib` is installed you get a heatmap; otherwise the table above already tells the story.

In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 6))
    im = ax.imshow(M, aspect="auto", cmap="viridis", vmin=0, vmax=1)
    ax.set_xticks(range(len(measure_names))); ax.set_xticklabels(measure_names, rotation=45, ha="right")
    ax.set_yticks(range(len(ids)))
    ax.set_yticklabels([f"{i} (T{t})" if t is not None else f"{i} (neg)" for i, t in zip(ids, types)], fontsize=7)
    ax.set_title("Similarity score by measure (rows = example pairs)")
    fig.colorbar(im, label="similarity"); plt.tight_layout(); plt.show()
except Exception as e:
    print("matplotlib not available -> skipping heatmap (%s)" % type(e).__name__)
    print("The printed matrix above contains the same information.")

## Where each measure goes blind

Look down the columns:

- **`ngram_cosine`** collapses on **Type-2** clones — renaming destroys the shared n-grams even though the logic is identical.
- **Every lexical measure** drops on **Type-4** clones (loop vs. recursion, or the closed-form `n*(n+1)/2`). Surface features simply are not there.
- **Token measures stay high on the hard negatives** (`sum` vs. `max`, shared file-loading boilerplate) — that is a **false positive** waiting to happen.
- **`metrics_cosine`** is ~1.0 for *everything* on these tiny snippets: it looks at gross shape only, so it cannot separate clones from non-clones here. A confident-looking but uninformative measure is dangerous on its own.

Let us quantify the *separation* each measure achieves: mean score on clones minus mean score on non-clones (bigger = better discrimination on these examples).

In [ ]:
clone_mask = labels == 1
neg_mask = labels == 0
print(f"{'measure':20s} {'mean@clone':>11s} {'mean@neg':>9s} {'separation':>11s}")
print("-" * 54)
sep = []
for j, m in enumerate(measure_names):
    mc = M[clone_mask, j].mean()
    mn = M[neg_mask, j].mean()
    sep.append((m, mc - mn))
    print(f"{m:20s} {mc:11.2f} {mn:9.2f} {mc-mn:11.2f}")
print("\nBest separators on these examples:")
for m, d in sorted(sep, key=lambda r: -r[1])[:3]:
    print(f"  {m:20s} {d:+.2f}")

## Complementarity: do measures make the *same* mistakes?

If all measures were highly correlated, combining them would add nothing. Let us compute the correlation between measures across the example pairs and find the **least correlated** pairs — those carry independent evidence and are the most valuable to combine.

In [ ]:
C = np.corrcoef(M.T)   # measure-by-measure correlation
# find the least-correlated distinct pairs
pairs_corr = []
for a in range(len(measure_names)):
    for b in range(a + 1, len(measure_names)):
        pairs_corr.append((measure_names[a], measure_names[b], C[a, b]))
pairs_corr.sort(key=lambda r: r[2])
print("Least-correlated measure pairs (most complementary):")
for a, b, c in pairs_corr[:6]:
    print(f"  {a:18s} <-> {b:18s}  r = {c:+.2f}")
print("\nMost-correlated (redundant) pairs:")
for a, b, c in pairs_corr[-4:]:
    print(f"  {a:18s} <-> {b:18s}  r = {c:+.2f}")

### Takeaway

No single measure is strong on **all** clone types *and* robust to **all** hard negatives. Crucially, the measures are **complementary** — they fail on different inputs. That is precisely the condition under which an **ensemble** outperforms its members.

### Exercise
Pick the two least-correlated measures above. By hand, design a rule that combines them to correctly classify *all* the example pairs. Is it possible with two measures? Three?

**Next:** [`03_ensemble_and_interpretability.ipynb`](03_ensemble_and_interpretability.ipynb) — combine the measures, evaluate on the repo's **real** benchmark data, and explain a single decision.